### EDA

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

temp_dir = Path("/home/kcv/Desktop/Rate_Capability/Data/temperature")
sample_path = sorted(temp_dir.glob("*.txt"))[0]  # pick a file
print("Analyzing:", sample_path.name)

# adjust delimiter/headers as needed; many txt logs are tab- or whitespace-separated
df = pd.read_csv(sample_path, sep=None, engine="python")  # infers delimiter
# if the file has no header row, add names: df = pd.read_csv(sample_path, sep="\t", header=None, names=[...])

display(df.head())
print(df.info())
print(df.describe(include="all"))
print("Missing-value ratio:\n", df.isna().mean())

Analyzing: Temp_Test_File_nano1_0043.txt


,Nano1,C_LFP_REPT_1_150_0043,2025-11-05 12:04:05,17.51,17.25,17.25.1,17.51.1
0,Nano1,C_LFP_REPT_1_150_0043,2025-11-05 12:04:06,17.42,17.34,17.59,17.59
1,Nano1,C_LFP_REPT_1_150_0043,2025-11-05 12:04:07,17.59,16.99,17.34,16.99
2,Nano1,C_LFP_REPT_1_150_0043,2025-11-05 12:04:08,17.25,17.08,17.85,17.34
3,Nano1,C_LFP_REPT_1_150_0043,2025-11-05 12:04:09,17.51,17.25,16.99,17.42
4,Nano1,C_LFP_REPT_1_150_0043,2025-11-05 12:04:10,17.42,17.16,17.68,17.34


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 587350 entries, 0 to 587349
Data columns (total 7 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Nano1                  587350 non-null  object 
 1   C_LFP_REPT_1_150_0043  587350 non-null  object 
 2   2025-11-05 12:04:05    587350 non-null  object 
 3   17.51                  587350 non-null  float64
 4   17.25                  587350 non-null  float64
 5   17.25.1                587350 non-null  float64
 6   17.51.1                587350 non-null  float64
dtypes: float64(4), object(3)
memory usage: 31.4+ MB
None
         Nano1  C_LFP_REPT_1_150_0043  2025-11-05 12:04:05          17.51  \
count   587350                 587350               587350  587350.000000   
unique       1                      1               587350            NaN   
top      Nano1  C_LFP_REPT_1_150_0043  2025-11-05 12:04:06            NaN   
freq    587350                 587350          

### Merge with Capacity data 

In [4]:
from pathlib import Path
from utils.temperature_merge import merge_temperature_directory

source = Path("/home/kcv/Desktop/Rate_Capability/Data/temperature")
output = Path("/home/kcv/Desktop/Rate_Capability/results/data/merged_temperarure_data")
cells = Path("/home/kcv/Desktop/Rate_Capability/Data")  # RD_RateCapability_*.csv files

paths = merge_temperature_directory(source, output, cell_data_dir=cells, tolerance="5s")
print(f"Merged {len(paths)} cells → {output}")


Merged 2 cells → /home/kcv/Desktop/Rate_Capability/results/data/merged_temperarure_data


In [5]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

merged_dir = Path("/home/kcv/Desktop/Rate_Capability/results/data/merged_temperarure_data")
output_pdf = Path("/home/kcv/Desktop/Rate_Capability/results/plots/temperature_timeseries.pdf")
output_pdf.parent.mkdir(parents=True, exist_ok=True)

csv_paths = sorted(merged_dir.glob("*_temperature.csv"))
if not csv_paths:
    raise FileNotFoundError(f"No *_temperature.csv files in {merged_dir}")

import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap

with PdfPages(output_pdf) as pdf:
    for path in csv_paths:
        df = pd.read_csv(path, parse_dates=["absolute time"])
        temp_cols = [c for c in df.columns if c.startswith("T_")]
        if not temp_cols:
            continue

        cmap = get_cmap("tab10", len(temp_cols))
        for idx, col in enumerate(temp_cols):
            fig, ax = plt.subplots(figsize=(10, 4))
            color = cmap(idx)
            ax.plot(df["absolute time"], df[col], color=color, linewidth=1.3)
            ax.set_title(f"{path.stem} – {col}")
            ax.set_xlabel("Absolute time")
            ax.set_ylabel("Temperature (°C)")
            fig.autofmt_xdate()
            fig.tight_layout()
            pdf.savefig(fig)
            plt.close(fig)


print(f"Saved all temperature plots to {output_pdf}")


/tmp/ipykernel_45033/4003467693.py:24: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("tab10", len(temp_cols))
/tmp/ipykernel_45033/4003467693.py:24: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("tab10", len(temp_cols))


Saved all temperature plots to /home/kcv/Desktop/Rate_Capability/results/plots/temperature_timeseries.pdf


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.colors as mcolors
from scipy.signal import savgol_filter

BIN = 0.01  # 10 mV bins
SG_WINDOW = 15  # must be odd; adjusted automatically if needed
SG_POLY = 2
MERGED_FILE = Path("/home/kcv/Desktop/Rate_Capability/results/data/merged_temperarure_data/C_LFP_REPT_1_150_0043_temperature.csv")
PLOTS_DIR = Path("/home/kcv/Desktop/Rate_Capability/results/plots")
DATA_DIR = Path("/home/kcv/Desktop/Rate_Capability/results/data")
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
PDF_PATH = PLOTS_DIR / "dTdV_Charge_Discharge_by_Crate.pdf"
SUMMARY_PATH = DATA_DIR / "dTdV_peak_summary.csv"

df = pd.read_csv(MERGED_FILE, parse_dates=["absolute time"])
df["volt(v)"] = pd.to_numeric(df["volt(v)"], errors="coerce")
df["current(a)"] = pd.to_numeric(df["current(a)"], errors="coerce")
df["cycle no"] = pd.to_numeric(df.get("cycle no"), errors="coerce")
df["step no"] = pd.to_numeric(df.get("step no"), errors="coerce")
df["step name"] = df["step name"].astype(str).fillna("")
temp_cols = [c for c in df.columns if c.startswith("T_")]

def step_category(name: str) -> str:
    name = name.lower()
    if "dchg" in name or "dis" in name:
        return "Discharge"
    if "chg" in name:
        return "Charge"
    return "Other"

def compute_dtdv(step_df, temp_col):
    temp = pd.to_numeric(step_df[temp_col], errors="coerce")
    data = step_df.assign(temp=temp).dropna(subset=["temp"])
    grouped = (
        data.groupby("volt_bin")
        .agg({"volt(v)": "mean", "temp": "mean", "c_rate": "mean"})
        .reset_index(drop=True)
    )
    if len(grouped) < 2:
        return None
    grouped["dV"] = grouped["volt(v)"].diff()
    grouped["dT"] = grouped["temp"].diff()
    grouped["dT_dV"] =abs(grouped["dT"] / grouped["dV"]) 
    result = grouped.dropna(subset=["dT_dV"]).copy()
    if result.empty:
        return None

    # Savitzky–Golay smoothing on dT/dV
    y = result["dT_dV"].to_numpy()
    window = SG_WINDOW if SG_WINDOW % 2 == 1 else SG_WINDOW + 1
    window = min(window, len(y))
    if window % 2 == 0:
        window -= 1
    if window >= SG_POLY + 2:
        result["dT_dV_smooth"] = savgol_filter(y, window, SG_POLY)
    else:
        result["dT_dV_smooth"] = y
    return result

summary_rows = []

with PdfPages(PDF_PATH) as pdf:
    for temp_col in temp_cols:
        for cycle, cycle_df in df.groupby("cycle no"):
            if pd.isna(cycle):
                continue

            fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
            plotted = False

            for label, ax in zip(["Charge", "Discharge"], axes):
                subset = cycle_df[cycle_df["step name"].apply(step_category) == label]
                ax.set_title(f"{label} – Cycle {int(cycle)} – {temp_col}")
                ax.set_xlabel("Voltage (V)")
                if label == "Charge":
                    ax.set_ylabel("dT/dV (°C/V)")
                ax.grid(alpha=0.3)

                legend_entries = []

                for (step_no, step_name), step_df in subset.groupby(["step no", "step name"]):
                    step_df = step_df.dropna(subset=["volt(v)", "current(a)"]).sort_values("absolute time")
                    if step_df.empty:
                        continue

                    c_rate_step = round(step_df["current(a)"].abs().max() / 150.0, 2)
                    step_df = step_df.copy()
                    step_df["c_rate"] = c_rate_step
                    step_df["volt_bin"] = (np.floor(step_df["volt(v)"] / BIN) * BIN).round(3)

                    result = compute_dtdv(step_df, temp_col)
                    if result is None:
                        continue

                    legend_entries.append((c_rate_step, step_no, step_name, result))

                    peak_idx = result["dT_dV_smooth"].abs().idxmax()
                    summary_rows.append(
                        {
                            "cycle": int(cycle),
                            "step no": int(step_no),
                            "step name": step_name,
                            "temperature_col": temp_col,
                            "c_rate": c_rate_step,
                            "peak_dT/dV": result.loc[peak_idx, "dT_dV_smooth"],
                            "voltage_at_peak": result.loc[peak_idx, "volt(v)"],
                        }
                    )

                legend_entries.sort(key=lambda x: x[0])
                if not legend_entries:
                    continue

                c_rates = [entry[0] for entry in legend_entries]
                cmap = mcolors.ListedColormap(plt.cm.viridis(np.linspace(0, 1, len(legend_entries))))
                norm = mcolors.Normalize(vmin=min(c_rates), vmax=max(c_rates))
                handles = []

                for idx, (c_rate_step, step_no, step_name, result) in enumerate(legend_entries):
                    color = plt.cm.viridis(norm(c_rate_step))
                    ax.scatter(
                        result["volt(v)"],
                        result["dT_dV_smooth"],
                        color=color,
                        s=5,
                        alpha=0.85,
                        linewidths=0,
                    )
                    pts = result[["volt(v)", "dT_dV_smooth"]].to_numpy()
                    if len(pts) > 1:
                        segments = np.stack([pts[:-1], pts[1:]], axis=1)
                        ax.add_collection(LineCollection(segments, colors=color, linewidths=0.9, alpha=0.9))

                    label_text = f"{c_rate_step:.2f} C – step {int(step_no)}"
                    handles.append(
                        plt.Line2D([0], [0], marker="o", color=color, linestyle="", label=label_text, markersize=4)
                    )
                    plotted = True

                if handles:
                    ax.legend(handles=handles, title="C-rate / Step", fontsize=6, frameon=False, loc="best")

            if plotted:
                fig.suptitle(MERGED_FILE.stem)
                fig.tight_layout()
                pdf.savefig(fig, dpi=120)
            plt.close(fig)

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SUMMARY_PATH, index=False)
print(f"Saved dT/dV plots to {PDF_PATH}")
print(f"Wrote summary to {SUMMARY_PATH}")


Saved dT/dV plots to /home/kcv/Desktop/Rate_Capability/results/plots/dTdV_Charge_Discharge_by_Crate.pdf
Wrote summary to /home/kcv/Desktop/Rate_Capability/results/data/dTdV_peak_summary.csv


# Heat Generation

In [6]:
ADR_DIR = Path("/home/kcv/Desktop/Rate_Capability/results/data/adr_data")
cell_name = "RD_RateCapability_0001"

df = pd.read_csv(ADR_DIR / f"{cell_name}_adr_data.csv")
df

,cycle no,step no,step name,c_rate,current(a),volt(v),OCV,soc(%),capacity(ah),resistance(ohm),voltage_sag(v),apparent_direct_resistance(ohm),voltage_loss_ratio
0,1,1,Rest,0.0,0.0,3.405835,2.799843,0.0,0.0,NaN,-0.605992,NaN,-0.216438
1,1,1,Rest,0.0,0.0,3.405824,2.799843,0.0,0.0,NaN,-0.605981,NaN,-0.216434
2,1,1,Rest,0.0,0.0,3.405815,2.799843,0.0,0.0,NaN,-0.605972,NaN,-0.216431
3,1,1,Rest,0.0,0.0,3.405832,2.799843,0.0,0.0,NaN,-0.605989,NaN,-0.216437
4,1,1,Rest,0.0,0.0,3.405825,2.799843,0.0,0.0,NaN,-0.605982,NaN,-0.216434
...,...,...,...,...,...,...,...,...,...,...,...,...,...
569471,3,23,Rest,0.0,0.0,2.789389,2.799843,0.0,0.0,NaN,0.010454,NaN,0.003734
569472,3,23,Rest,0.0,0.0,2.789410,2.799843,0.0,0.0,NaN,0.010433,NaN,0.003726
569473,3,23,Rest,0.0,0.0,2.789407,2.799843,0.0,0.0,NaN,0.010436,NaN,0.003727
569474,3,23,Rest,0.0,0.0,2.789416,2.799843,0.0,0.0,NaN,0.010427,NaN,0.003724


In [ ]:
from pathlib import Path
import pandas as pd

ADR_DIR = Path("/home/kcv/Desktop/Rate_Capability/results/data/adr_data")
OUT_DIR = Path("/home/kcv/Desktop/Rate_Capability/results/data/heat_generation")
OUT_DIR.mkdir(parents=True, exist_ok=True)

first_entries = set()

adr_paths = sorted(ADR_DIR.glob("*_adr_data.csv"))
if not adr_paths:
    raise FileNotFoundError(f"No *_adr_data.csv files in {ADR_DIR}")

for path in adr_paths:
    df = pd.read_csv(path)
    df["current(a)"] = pd.to_numeric(df.get("current(a)"), errors="coerce")
    df["apparent_direct_resistance(ohm)"] = pd.to_numeric(
        df.get("apparent_direct_resistance(ohm)"), errors="coerce"
    )

    df["heat_generation(Q)"] = df["current(a)"] ** 2 / df["apparent_direct_resistance(ohm)"]
    df.loc[df["apparent_direct_resistance(ohm)"] == 0, "heat_generation(Q)"] = pd.NA

    out_path = OUT_DIR / (path.stem.replace("_adr_data", "") + "_heat_generation.csv")
    df.to_csv(out_path, index=False)
    print(f"Wrote {out_path.name}")

print(f"Saved {len(adr_paths)} heat-generation files to {OUT_DIR}")


In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

HEAT_DIR = Path("/home/kcv/Desktop/Rate_Capability/results/data/heat_generation")
PLOTS_DIR = Path("/home/kcv/Desktop/Rate_Capability/results/plots")
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
pdf_path = PLOTS_DIR / "heat_vs_capacity_crate.pdf"

heat_paths = sorted(HEAT_DIR.glob("*_heat_generation.csv"))
if not heat_paths:
    raise FileNotFoundError(f"No heat-generation files in {HEAT_DIR}")

with PdfPages(pdf_path) as pdf:
    for path in heat_paths:
        df = pd.read_csv(path)

        # Ensure numeric columns
        cap = pd.to_numeric(df.get("capacity(ah)"), errors="coerce")
        heat = pd.to_numeric(df.get("heat_generation(Q)"), errors="coerce")
        crate = pd.to_numeric(df.get("c_rate"), errors="coerce")

        mask = cap.notna() & heat.notna()
        if not mask.any():
            continue

        fig, ax = plt.subplots(figsize=(8, 3.5))
        sc = ax.scatter(
            cap[mask],
            heat[mask],
            c=crate[mask],
            cmap="viridis",
            s=4,
            alpha=0.75,
            linewidths=0,
        )
        ax.set_xlabel("Capacity (Ah)")
        ax.set_ylabel("Heat generation Q (A²·Ω⁻¹)")
        ax.set_title(path.stem)
        ax.grid(alpha=0.3)

        if crate[mask].notna().any():
            cbar = fig.colorbar(sc, ax=ax, pad=0.02)
            cbar.set_label("C-rate")

        fig.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

print(f"Saved heat vs capacity plots to {pdf_path}")


Saved heat vs capacity plots to /home/kcv/Desktop/Rate_Capability/results/plots/heat_vs_capacity_crate.pdf
